# AIC2024 KIS — Phase 2: CLIP embedding + FAISS (Kaggle)

Notebook **self-contained**: không cần source repo, chỉ cần Kaggle Dataset chứa `artifacts/`.

## Chuẩn bị
1. Zip thư mục `artifacts/` (gồm `metadata.sqlite` + `keyframes/`) và upload thành Kaggle Dataset.
2. Trong Notebook Settings: bật **Accelerator → GPU** và **Internet → On** (để tải CLIP ViT-B/16).
3. Add dataset vào notebook, chạy toàn bộ cell.
4. Tải `artifacts_embed.zip` ở `/kaggle/working/`, giải nén vào `artifacts/` local.

## Output
`embeddings.f16.npy`, `id_map.npy`, `faiss.index`, `manifest.json`.

Notebook tự resume nếu session restart: tiến độ lưu ở `/kaggle/working/embed/progress.json`.


## 1. Cấu hình


In [ ]:
import os
from pathlib import Path

MODEL_NAME = "sentence-transformers/clip-ViT-B-16"
DIM = 512
BATCH_SIZE = 128

ARTIFACTS_DIR = os.environ.get("EMBED_ARTIFACTS_DIR") or None
WORK_DIR = Path(os.environ.get("EMBED_WORK_DIR", "/kaggle/working/embed"))
ZIP_PATH = Path(os.environ.get("EMBED_ZIP_PATH", "/kaggle/working/artifacts_embed.zip"))
RESUME = os.environ.get("EMBED_RESUME", "1") != "0"
MAX_KEYFRAMES = (
    int(os.environ["EMBED_MAX_KEYFRAMES"])
    if os.environ.get("EMBED_MAX_KEYFRAMES")
    else None
)

print("model:", MODEL_NAME, "| dim:", DIM, "| batch:", BATCH_SIZE)
print("artifacts override:", ARTIFACTS_DIR)
print("work:", WORK_DIR)
print("zip:", ZIP_PATH)


## 2. Cài dependency

Kaggle đã có `torch`; cell này chỉ bổ sung `sentence-transformers` và `faiss-cpu`.


In [ ]:
import subprocess
import sys

if os.environ.get("EMBED_SKIP_INSTALL") != "1":
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "sentence-transformers", "faiss-cpu"],
        check=True,
    )
else:
    print("skip install (EMBED_SKIP_INSTALL=1)")


## 3. Tìm `artifacts/`

Tự dò `metadata.sqlite` + `keyframes/` trong `/kaggle/input`. Nếu không thấy, set biến môi trường
`EMBED_ARTIFACTS_DIR` hoặc sửa trực tiếp `ARTIFACTS_DIR` ở cell 1.


In [ ]:
import shutil
import sqlite3

PATTERNS = (
    "*/artifacts/metadata.sqlite",
    "*/metadata.sqlite",
    "*/*/artifacts/metadata.sqlite",
    "*/*/metadata.sqlite",
    "*/*/*/artifacts/metadata.sqlite",
)


def candidate_dirs() -> list[Path]:
    dirs: list[Path] = []
    if ARTIFACTS_DIR:
        dirs.append(Path(ARTIFACTS_DIR))
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for pattern in PATTERNS:
            dirs.extend(db.parent for db in input_root.glob(pattern))
        if not dirs:
            dirs.extend(db.parent for db in input_root.rglob("metadata.sqlite"))
    local = Path.cwd() / "artifacts"
    if local.is_dir():
        dirs.append(local)
    return dirs


candidates = []
seen = set()
for path in candidate_dirs():
    resolved = path.resolve()
    if resolved in seen:
        continue
    seen.add(resolved)
    if (resolved / "metadata.sqlite").is_file() and (resolved / "keyframes").is_dir():
        candidates.append(resolved)

if not candidates:
    raise FileNotFoundError(
        "Không tìm thấy metadata.sqlite + keyframes/. "
        "Set EMBED_ARTIFACTS_DIR hoặc kiểm tra lại Kaggle dataset."
    )

ARTIFACTS_DIR = candidates[0]
SOURCE_DB_PATH = ARTIFACTS_DIR / "metadata.sqlite"

WORK_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = WORK_DIR / "metadata.sqlite"
for suffix in ("", "-wal"):
    source = Path(str(SOURCE_DB_PATH) + suffix)
    target = Path(str(DB_PATH) + suffix)
    if source.is_file():
        shutil.copy2(source, target)
    elif suffix:
        target.unlink(missing_ok=True)
Path(str(DB_PATH) + "-shm").unlink(missing_ok=True)

print("artifacts:", ARTIFACTS_DIR)
print("db source:", SOURCE_DB_PATH)
print("db working copy:", DB_PATH)
if len(candidates) > 1:
    print("bỏ qua candidate khác:", [str(p) for p in candidates[1:]])

conn = sqlite3.connect(DB_PATH)
try:
    video_rows = conn.execute(
        "SELECT video_id, generation, duration_ms FROM videos ORDER BY video_id"
    ).fetchall()
finally:
    conn.close()
print(f"videos trong DB: {len(video_rows):,}")


## 4. Đọc danh sách keyframe của generation hiện tại

Chỉ lấy keyframe khớp `videos.generation`; thứ tự cố định theo `kf_uid` (trùng thứ tự row FAISS).
Cell dừng ngay nếu có file ảnh thiếu/rỗng.


In [ ]:
conn = sqlite3.connect(DB_PATH)
try:
    rows = conn.execute(
        """
        SELECT k.kf_uid, k.video_id, k.generation, k.shot_idx, k.kf_idx, k.ts_ms, k.path
        FROM keyframes k
        JOIN videos v ON v.video_id = k.video_id AND k.generation = v.generation
        ORDER BY k.kf_uid
        """
    ).fetchall()
finally:
    conn.close()

if MAX_KEYFRAMES:
    rows = rows[:MAX_KEYFRAMES]
if not rows:
    raise RuntimeError("Không có keyframe nào ở generation hiện tại.")

missing = []
for row in rows:
    image_path = ARTIFACTS_DIR / row[6]
    try:
        if image_path.stat().st_size == 0:
            missing.append(str(image_path))
    except OSError:
        missing.append(str(image_path))

print(f"keyframes: {len(rows):,}")
print(f"videos: {len({row[1] for row in rows}):,}")
print(f"missing/empty: {len(missing):,}")
if missing:
    raise FileNotFoundError("Thiếu file keyframe:\n" + "\n".join(missing[:20]))


## 5. Chuẩn bị file embedding + resume

`embeddings.f16.npy` là memmap `(N, 512)` float16. Tiến độ và hash kế hoạch được lưu cạnh đó;
chạy lại notebook sẽ tiếp tục từ batch dở dang thay vì embed lại từ đầu.


In [ ]:
import hashlib
import json

import numpy as np

uids = np.asarray([row[0] for row in rows], dtype=np.int64)
plan_hash = hashlib.sha256(MODEL_NAME.encode() + b"\n" + uids.tobytes()).hexdigest()

emb_path = WORK_DIR / "embeddings.f16.npy"
progress_path = WORK_DIR / "progress.json"
plan_path = WORK_DIR / "plan.json"


def write_json(path: Path, payload: dict) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)


WORK_DIR.mkdir(parents=True, exist_ok=True)
total = len(rows)
expected_shape = (total, DIM)
rows_done = 0

if RESUME and emb_path.is_file():
    try:
        plan = json.loads(plan_path.read_text(encoding="utf-8"))
        progress = json.loads(progress_path.read_text(encoding="utf-8"))
        probe = np.lib.format.open_memmap(str(emb_path), mode="r")
        shape_ok = probe.shape == expected_shape and probe.dtype == np.float16
        del probe
        if plan.get("plan_hash") == plan_hash and shape_ok:
            rows_done = min(max(0, int(progress.get("rows_done", 0))), total)
    except (OSError, ValueError, KeyError):
        rows_done = 0

if not RESUME or rows_done == 0:
    emb_path.unlink(missing_ok=True)
    memmap = np.lib.format.open_memmap(
        str(emb_path), mode="w+", dtype=np.float16, shape=expected_shape
    )
    del memmap

write_json(
    plan_path,
    {
        "model": MODEL_NAME,
        "dim": DIM,
        "count": total,
        "max_keyframes": MAX_KEYFRAMES,
        "order": "kf_uid",
        "plan_hash": plan_hash,
    },
)
write_json(progress_path, {"rows_done": rows_done, "status": "running"})
print(f"resume from {rows_done:,}/{total:,}")


## 6. Load CLIP ViT-B/16


In [ ]:
import torch
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print("device:", DEVICE, "|", torch.cuda.get_device_name(0))
else:
    print("device:", DEVICE)

model = SentenceTransformer(MODEL_NAME, device=DEVICE)
model_dim = (
    model.get_embedding_dimension()
    if hasattr(model, "get_embedding_dimension")
    else model.get_sentence_embedding_dimension()
)
if model_dim != DIM:
    raise RuntimeError(f"model dim {model_dim} != configured {DIM}")
print("embedding dim:", model_dim)


## 7. Embed keyframe

Ảnh được decode bằng PIL, embed theo batch, L2-normalize, ghi thẳng vào memmap float16.
Ghi progress sau mỗi batch để resume được.


In [ ]:
import time

from PIL import Image

memmap = np.lib.format.open_memmap(str(emb_path), mode="r+")
started = time.time()
last_report = started

for start in range(rows_done, total, BATCH_SIZE):
    chunk = rows[start : start + BATCH_SIZE]
    images = []
    for row in chunk:
        with Image.open(ARTIFACTS_DIR / row[6]) as image:
            images.append(image.convert("RGB"))

    vectors = model.encode(
        images,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    vectors = np.asarray(vectors, dtype=np.float32)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    vectors = vectors / np.maximum(norms, 1e-12)

    memmap[start : start + len(chunk)] = vectors.astype(np.float16)
    memmap.flush()

    done = start + len(chunk)
    write_json(progress_path, {"rows_done": done, "status": "running"})

    now = time.time()
    if now - last_report >= 10 or done == total:
        elapsed = max(now - started, 1e-6)
        rate = (done - rows_done) / elapsed
        eta = (total - done) / rate if rate > 0 else 0
        print(
            f"{done:,}/{total:,}  {rate:,.1f} img/s  eta {eta / 60:,.1f} min",
            flush=True,
        )
        last_report = now

memmap.flush()
del memmap
write_json(progress_path, {"rows_done": total, "status": "done"})
print(f"done: {total:,} embeddings in {(time.time() - started) / 60:,.1f} min")


## 8. Build FAISS `IndexFlatIP`

Đọc lại memmap theo block, chuẩn hoá L2 lần nữa rồi add vào index. Assert
`index.ntotal == số keyframe` trước khi ghi file.


In [ ]:
import faiss

memmap = np.lib.format.open_memmap(str(emb_path), mode="r")
index = faiss.IndexFlatIP(DIM)
block_size = 50_000
for start in range(0, total, block_size):
    block = np.asarray(memmap[start : start + block_size], dtype=np.float32)
    faiss.normalize_L2(block)
    index.add(block)
del memmap

if index.ntotal != total:
    raise RuntimeError(f"FAISS ntotal {index.ntotal} != keyframes {total}")

faiss.write_index(index, str(WORK_DIR / "faiss.index"))
np.save(WORK_DIR / "id_map.npy", uids)
print("faiss:", type(index).__name__, "| ntotal:", index.ntotal, "| dim:", index.d)


## 9. Manifest + checksum


In [ ]:
import hashlib
from datetime import datetime, timezone


def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


manifest = {
    "version": 1,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "model": MODEL_NAME,
    "dim": DIM,
    "count": int(total),
    "dtype": "float16",
    "normalized": "l2",
    "faiss": {"type": "IndexFlatIP", "ntotal": int(index.ntotal)},
    "source": {
        "db": str(SOURCE_DB_PATH),
        "db_working_copy": str(DB_PATH),
        "keyframes_root": str(ARTIFACTS_DIR / "keyframes"),
        "max_keyframes": MAX_KEYFRAMES,
    },
    "order": "kf_uid",
    "plan_hash": plan_hash,
    "files": {
        "embeddings": "embeddings.f16.npy",
        "faiss": "faiss.index",
        "id_map": "id_map.npy",
    },
    "checksums": {
        "embeddings_sha256": sha256_file(WORK_DIR / "embeddings.f16.npy"),
        "faiss_sha256": sha256_file(WORK_DIR / "faiss.index"),
        "id_map_sha256": sha256_file(WORK_DIR / "id_map.npy"),
    },
}
write_json(WORK_DIR / "manifest.json", manifest)
print(json.dumps(manifest, indent=2, ensure_ascii=False))


## 10. Đóng gói output

Zip chỉ chứa artifact embedding (không chứa lại keyframes). Giải nén vào `artifacts/` local.


In [ ]:
import zipfile

README = """AIC2024 KIS - Phase 2 artifacts
Files:
  embeddings.f16.npy  (count x dim, float16, L2-normalized, row order = id_map)
  id_map.npy          (int64 kf_uid per row)
  faiss.index         (IndexFlatIP, same row order)
  manifest.json
Extract into the local `artifacts/` directory.
"""

ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
) as archive:
    for name in ("embeddings.f16.npy", "faiss.index", "id_map.npy", "manifest.json"):
        archive.write(WORK_DIR / name, arcname=name)
    archive.writestr("README.txt", README)

print(f"zip: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:,.1f} MB)")


## 11. Sanity check


In [ ]:
query = "a person riding a bicycle"
qvec = model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
qvec = np.asarray(qvec, dtype=np.float32)
faiss.normalize_L2(qvec)
scores, indices = index.search(qvec, 5)

print("query:", query)
for score, idx in zip(scores[0], indices[0]):
    if idx < 0:
        continue
    row = rows[int(idx)]
    print(f"  {score:.4f}  {row[1]}  t={row[5]}ms  {row[6]}")


## Sau khi chạy

1. Tải `artifacts_embed.zip` từ tab **Output** của Kaggle.
2. Giải nén vào `artifacts/` ở repo local.
3. Kiểm tra `artifacts/manifest.json` có `count` đúng bằng số keyframe và `faiss.ntotal == count`.
